# Boosting: Gradient Boosting and XGBoost

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209, [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/machine_learning.html)
- [Syllabus](https://joannabieri.com/machinelearning/IntroMachineLearning.pdf)

:::{.callout-important icon=false}
## How to use these notes

Two kinds of box show up in these notes.

**Blue Q boxes** are questions for you to answer **by hand, in a notebook, with a pen.** Not because I am old fashioned. Writing something down by hand is slow, and slow is the point: it is very hard to write an explanation you do not actually understand. You are welcome to use AI in this class for the mechanics of code, but these boxes are the part where you do the thinking yourself. Bring your written notes to class, I will ask to see them.

**Green You Try boxes** are optional code for you to work through. Nothing is collected and nothing is graded. They are there because you will learn more from changing a number and rerunning than from watching me do it.

**Every code cell begins with a tag** that tells you what to do with it.

- `# RUN THIS.` Setup, loading data, a plot. Copy it, run it, move on. You do not need to be able to write it from memory.
- `# LEARN TO WRITE THIS.` The pattern of the day. The homework will ask you for it, and so will the exam. Type it out yourself at least once rather than pasting it.
- `# DEMO ONLY.` Fake data or a contrived experiment that exists to show one idea. You would never write this for a real project and you do not need to be able to.

Short answers to the Q boxes are in drop down boxes at the very bottom. Write yours first.
:::


**Reading:** Geron chapter 6, the sections on **Boosting** (both **AdaBoost** and **Gradient Boosting**) and **Stacking**.

Day 6 built a crowd by training a lot of trees **at the same time** and letting them vote. Every tree saw its own bootstrap sample, nobody was in charge, and the average cancelled out their mistakes.

Today we build a crowd the other way: **one model at a time, each one fixing what the last one got wrong.** That is **boosting**, and it is the other half of Geron chapter 6. It is also, for tables of numbers like our wine, the family that usually wins competitions.

It comes with a catch that bagging did not have, and we will meet it early: **with boosting, more trees can make the model worse.**

Same wine and the same splits as Day 5 and Day 6. One change: **the test set stays closed today.** We opened it on Day 6 to report the forest, and it does not get opened again just because we have a new model. Everything today happens on validation and on cross validation.

# Setup

In [ ]:
# RUN THIS. The same wine and the same splits as Day 5 and Day 6.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

wine = pd.read_csv("data/winequality-red.csv", sep=";")
wine["good"] = (wine["quality"] >= 7).astype(int)

X = wine.drop(columns=["quality", "good"])
y = wine["good"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)

print("training rows:  ", X_train.shape[0])
print("validation rows:", X_valid.shape[0])
print("test rows:      ", X_test.shape[0], "  (closed today)")

---

# What Boosting Does

Here is the whole idea, in words, for **gradient boosting**.

1. Start with one small tree. It gets a lot wrong.
2. Look at what is left over, the part of the answer the model has not explained yet. That leftover is called the **residual**.
3. Train a new small tree **on the leftovers**, and add a fraction of it to the model.
4. Repeat, hundreds of times. Each new tree works on what the current model still gets wrong.

Nobody votes. The models are **added together**, and each one is a small correction to the ones before it.

This is easiest to see on a made up curve with one feature, so we can plot the model itself. Below, the blue dots are training points and the red line is the boosted model after 1 tree, after 25, and after 500.

In [ ]:
# DEMO ONLY. A fake curve with noise, so we can watch boosting build a model.
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)
x = np.sort(rng.uniform(-3, 3, 120))                    # 120 points between -3 and 3
y_fake = np.sin(x) + 0.3 * x + rng.normal(0, 0.35, 120)  # a curve, plus noise
X_fake = x.reshape(-1, 1)                                # sklearn wants a column, not a row

X_fake_train, X_fake_valid, y_fake_train, y_fake_valid = train_test_split(
    X_fake, y_fake, test_size=0.35, random_state=0)

grid = np.linspace(-3, 3, 400).reshape(-1, 1)     # smooth x values, just for drawing the line

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)

for ax, n in zip(axes, [1, 25, 500]):
    model = GradientBoostingRegressor(
        n_estimators=n,        # how many trees to add, one after another
        learning_rate=0.1,     # how much of each new tree to add. More on this below
        max_depth=2,           # each tree is tiny on purpose
        random_state=42)
    model.fit(X_fake_train, y_fake_train)

    ax.plot(X_fake_train, y_fake_train, "b.", markersize=5, label="training points")
    ax.plot(grid, model.predict(grid), "r-", linewidth=2, label="the boosted model")
    ax.set_title(str(n) + " trees")
    ax.grid()
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("images/01-boosting-stages.png", dpi=150, bbox_inches="tight")
plt.show()

With **1 tree** the model is a couple of flat steps. It is a tree of depth 2, so that is all it can be. With **25 trees** it has found the shape of the curve. With **500 trees** it is chasing individual points, including the noise.

That last panel should worry you, and the next section measures it.

:::{.callout-note icon=false}
## Q1. Write this one out by hand

**a.** In your own words, what does each new tree in a boosted model get trained on? How is that different from each new tree in a random forest?

**b.** In the 500 tree panel, the red line passes very close to nearly every blue dot. Why is that bad, given how `y_fake` was built?

**c.** Bagging trains its trees at the same time and boosting trains them one after another. Which one can you speed up by using more cores of your computer? Why?
:::

---

# More Trees Can Make It Worse

On Day 6 we added trees to the forest and the score went up and then flattened out. Extra trees cost time and nothing else. **Boosting is not like that.** Each new tree is fitted to what is left over, so once the real pattern is used up, the new trees start fitting the noise.

In [ ]:
# DEMO ONLY. Training and validation error as we add trees to a boosted model.
from sklearn.metrics import mean_squared_error

tree_counts = [1, 2, 5, 10, 25, 50, 100, 200, 500, 1000, 2000]
train_errors = []
valid_errors = []

for n in tree_counts:
    model = GradientBoostingRegressor(n_estimators=n, learning_rate=0.1, max_depth=2, random_state=42)
    model.fit(X_fake_train, y_fake_train)

    train_rmse = mean_squared_error(y_fake_train, model.predict(X_fake_train)) ** 0.5
    valid_rmse = mean_squared_error(y_fake_valid, model.predict(X_fake_valid)) ** 0.5

    train_errors.append(train_rmse)
    valid_errors.append(valid_rmse)
    print("trees", n, "  training RMSE", round(train_rmse, 3), "  validation RMSE", round(valid_rmse, 3))

plt.figure(figsize=(6.5, 4))
plt.semilogx(tree_counts, train_errors, "b-o", linewidth=2, label="training error")
plt.semilogx(tree_counts, valid_errors, "g-o", linewidth=2, label="validation error")
plt.axhline(0.35, color="gray", linestyle="--", linewidth=1, label="the noise in the data")
plt.xlabel("number of trees (log scale)")
plt.ylabel("RMSE")
plt.title("With boosting, more trees can make it worse")
plt.grid()
plt.legend()
plt.savefig("images/02-too-many-trees.png", dpi=150, bbox_inches="tight")
plt.show()

Read the green line. Validation error falls to **0.362 at 50 trees**, which is about as good as it can get, because the noise we added has a standard deviation of 0.35. Then it climbs back up: **0.396 at 100 trees, 0.459 at 500, and 0.476 at 2000.** Meanwhile the blue line marches to **0.000**. At 2000 trees the model reproduces every training point exactly and is worse than useless on new points.

So `n_estimators` in boosting is not "bigger is safer" like it was for a forest. It is a real knob you have to set, and the way you set it is the validation set.

:::{.callout-note icon=false}
## Q2. Write this one out by hand

**a.** On Day 6 we said more trees never hurt a random forest. Explain why boosting is different, in terms of what each new tree is trained on.

**b.** The validation error bottoms out at 0.362 and the noise in the data has a standard deviation of 0.35. Why can no model, however clever, get the validation error much below 0.35 here?

**c.** Suppose you only looked at the training error in that table. What would you conclude about 2000 trees, and what would go wrong when you used the model?
:::

---

# The Learning Rate

There is a second knob, and it works against the first one. `learning_rate` says how much of each new tree to add. A small learning rate means every tree is a small correction, so you need many of them. A big one means each tree changes the model a lot.

In [ ]:
# DEMO ONLY. The same fake curve, different learning rates.
learning_rates = [0.01, 0.03, 0.1, 0.3, 1.0]
counts = [1, 5, 10, 25, 50, 100, 200, 500, 1000]

plt.figure(figsize=(6.5, 4))

for rate in learning_rates:
    errors = []
    for n in counts:
        model = GradientBoostingRegressor(n_estimators=n, learning_rate=rate, max_depth=2, random_state=42)
        model.fit(X_fake_train, y_fake_train)
        errors.append(mean_squared_error(y_fake_valid, model.predict(X_fake_valid)) ** 0.5)

    best = min(errors)
    best_n = counts[errors.index(best)]
    print("learning rate", rate, "  best at", best_n, "trees, validation RMSE", round(best, 3))

    plt.semilogx(counts, errors, "-o", linewidth=2, markersize=4, label="learning rate " + str(rate))

plt.axhline(0.35, color="gray", linestyle="--", linewidth=1, label="the noise in the data")
plt.xlabel("number of trees (log scale)")
plt.ylabel("validation RMSE")
plt.title("Small steps need more trees. Big steps overshoot.")
plt.grid()
plt.legend(fontsize=8)
plt.savefig("images/03-learning-rate.png", dpi=150, bbox_inches="tight")
plt.show()

Look at where each curve bottoms out.

| learning rate | best number of trees | best validation RMSE |
|---|---|---|
| 0.01 | 500 | 0.362 |
| 0.03 | 100 | 0.358 |
| 0.1 | 50 | 0.362 |
| 0.3 | 10 | 0.350 |
| 1.0 | 1 | 0.391 |

The first four all reach about the same error, they just need different numbers of trees to get there. Small learning rate, many trees. Big learning rate, few trees. At **1.0** the model takes steps so large that it never settles, and the best it manages is a single tree.

Two things follow, and they are how people actually use boosting.

1. **`learning_rate` and `n_estimators` are one decision, not two.** Lower one and you must raise the other.
2. Since a small learning rate with too many trees starts to overfit, you want to stop adding trees at the bottom of that green curve. Doing that automatically is called **early stopping**, and it is the next section.

:::{.callout-note icon=false}
## Q3. Write this one out by hand

**a.** Your model uses `learning_rate=0.1` with 50 trees. You want to try `learning_rate=0.01`. Roughly what should you do to `n_estimators`, and why?

**b.** Why does a learning rate of 1.0 do worse than every smaller rate here, even with 1000 trees available?

**c.** A friend says "I will just set `learning_rate` very small and `n_estimators` very large, that must be safest." What does the plot say happens, and what does it cost?
:::

---

# Boosting on the Wine

Enough fake curves. Here is gradient boosting as a classifier, on the wine, scored the Day 5 way on the same validation wines.

In [ ]:
# LEARN TO WRITE THIS. Gradient boosting on the wine.
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score

boosted = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=5,         # deeper than the fake demo. Wine has 11 features, not 1
    learning_rate=0.1,
    random_state=42)
boosted.fit(X_train, y_train)

y_prob_boost = boosted.predict_proba(X_valid)[:, 1]
print("gradient boosting, validation average precision:", round(average_precision_score(y_valid, y_prob_boost), 3))

**0.673.** Day 6's random forest got 0.667 on these same wines. Hold that thought, because the difference is smaller than it looks, and we come back to it in two sections.

There is an older boosting method in the reading too, **AdaBoost**. Instead of fitting leftovers, it re-weights the training rows: every row the current model gets wrong becomes more important for the next tree. It is usually built from **stumps**, trees with a single question.

In [ ]:
# RUN THIS. AdaBoost, the other method in the reading.
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1),   # a "stump": one question, two leaves
    n_estimators=200,
    random_state=42)
ada.fit(X_train, y_train)

print("AdaBoost, validation average precision:",
      round(average_precision_score(y_valid, ada.predict_proba(X_valid)[:, 1]), 3))

**0.518**, well behind. Stumps are a very weak starting point for this data, and AdaBoost is mostly of historical interest now. Gradient boosting replaced it, and the next section is what replaced gradient boosting in practice.

---

# XGBoost

**XGBoost** is gradient boosting, written for speed and with regularization built in. It is a separate package, not part of sklearn, and it is the tool people reach for on tables of numbers. It has won a lot of Kaggle competitions. The good news for you: it copies sklearn's interface, so `fit`, `predict` and `predict_proba` all work exactly as you expect.

In [ ]:
# LEARN TO WRITE THIS. XGBoost, which behaves like an sklearn model.
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss")    # what it watches while training. logloss is the usual default for classification
xgb_model.fit(X_train, y_train)

print("XGBoost, validation average precision:",
      round(average_precision_score(y_valid, xgb_model.predict_proba(X_valid)[:, 1]), 3))

**0.605** with 200 trees, and it fits in a fraction of a second.

Now the useful part. XGBoost can watch a validation set while it trains and **stop by itself** when the validation score stops improving. That is early stopping, the fix for the green curve in the second figure.

In [ ]:
# LEARN TO WRITE THIS. Let XGBoost decide how many trees to use.
xgb_early = xgb.XGBClassifier(
    n_estimators=2000,            # an upper limit, not a promise. It will stop long before this
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    eval_metric="aucpr",          # watch the precision/recall area, the Day 5 score for rare positives
    early_stopping_rounds=50)     # stop when 50 trees in a row fail to improve it

xgb_early.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],    # the set it watches. Never the test set
    verbose=False)

print("it stopped at tree:", xgb_early.best_iteration, "out of 2000")
print("validation average precision:",
      round(average_precision_score(y_valid, xgb_early.predict_proba(X_valid)[:, 1]), 3))

It stopped at tree **193** and scored **0.622**. We never had to guess `n_estimators`: we gave it a ceiling of 2000 and it found its own stopping point.

One thing to be clear about. The validation set is being used to make a choice here, which is exactly what a validation set is for. The test set is not involved, and the number you report at the end still has to come from data that had no say in any choice.

:::{.callout-note icon=false}
## Q4. Write this one out by hand

**a.** What does `early_stopping_rounds=50` actually mean? Why not stop the first time a tree fails to improve the score?

**b.** Why is it fine to early stop on the validation set, but not on the test set?

**c.** We gave `n_estimators=2000` and it used 193. Would setting `n_estimators=300` have changed the answer here? What about `n_estimators=100`?
:::

---

# Is the Boosted Model Actually Better?

Gradient boosting got **0.673** on validation and the Day 6 forest got **0.667**. That is a win of 0.006. Before we believe it, remember what the validation set is: **300 wines, 41 of them good.** Moving two or three wines up or down the ranking moves that number more than 0.006.

So compare them properly. `cross_val_score` with **repeated** stratified folds fits each model many times on different splits of the training data, and gives us a spread instead of one number.

In [ ]:
# LEARN TO WRITE THIS. Compare two models with repeated cross validation.
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier

cv = RepeatedStratifiedKFold(
    n_splits=5,        # 5 folds
    n_repeats=4,       # done 4 times with different shuffles, so 20 fits per model
    random_state=42)

forest = RandomForestClassifier(n_estimators=200, random_state=42)
boost = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)

for name, model in [("random forest", forest), ("gradient boosting", boost)]:
    scores = cross_val_score(model, X_train_full, y_train_full, cv=cv, scoring="average_precision")
    print(name)
    print("   mean average precision:", round(scores.mean(), 3))
    print("   standard deviation:    ", round(scores.std(), 3))
    print("   worst and best fold:   ", round(scores.min(), 3), "to", round(scores.max(), 3))

The forest averages **0.633** and the boosted model **0.611**, and both wobble by about **0.085** from fold to fold. The gap between the two models is a quarter of the noise in the measurement, and it points the **other way** from the single validation number.

The honest conclusion: **on this data, with these settings, the two models are tied.** Boosting did not beat the forest. A 0.006 lead on one validation set was never evidence of anything.

This is worth more than it looks. It is very easy to try six models, report the highest number, and believe it. The cure is cheap: compare with repeated cross validation, look at the spread, and only claim a winner when the gap is bigger than the wobble.

And notice what we did **not** do: open the test set to break the tie. We spent it on Day 6. If we scored every new model on it, it would stop being a test set and become another validation set.

:::{.callout-note icon=false}
## Q5. Write this one out by hand

**a.** Why is a mean over 20 fits more trustworthy than one validation score, even though both use data the model did not train on?

**b.** The standard deviation was about 0.085. Explain, to somebody who has not taken this class, why a difference of 0.006 between two models does not mean much next to that.

**c.** We have two tied models. Give one reason you might ship the forest and one reason you might ship the boosted model, neither of which is the score.
:::

---

# Where Boosting Does Win

Our wine is 1199 training rows and 11 features. That is small. Boosting tends to pull ahead on bigger tables, and it gets there faster. Here it is on the bank marketing data from Weekly Homework 3, with `duration` dropped because it leaks.

This cell takes longer than anything else today, about fifteen seconds.

In [ ]:
# RUN THIS. Boosting against the forest on 45,211 bank customers.
import time

bank = pd.read_csv("../WeeklyHomework/data/bank-full.csv", sep=";")   # the Weekly Homework 3 file
y_bank = (bank["y"] == "yes").astype(int)
X_bank = pd.get_dummies(bank.drop(columns=["y", "duration"]), dtype=int)   # duration is the leak

Xb_train_full, Xb_test, yb_train_full, yb_test = train_test_split(
    X_bank, y_bank, test_size=0.25, stratify=y_bank, random_state=42)
Xb_train, Xb_valid, yb_train, yb_valid = train_test_split(
    Xb_train_full, yb_train_full, test_size=0.25, stratify=yb_train_full, random_state=42)

start = time.time()
bank_forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42)
bank_forest.fit(Xb_train, yb_train)
forest_seconds = time.time() - start
forest_ap = average_precision_score(yb_valid, bank_forest.predict_proba(Xb_valid)[:, 1])
print("forest:  ", round(forest_seconds, 1), "seconds,  validation average precision", round(forest_ap, 3))

start = time.time()
bank_boost = xgb.XGBClassifier(n_estimators=3000, learning_rate=0.05, max_depth=3,
                               random_state=42, eval_metric="aucpr", early_stopping_rounds=50)
bank_boost.fit(Xb_train, yb_train, eval_set=[(Xb_valid, yb_valid)], verbose=False)
boost_seconds = time.time() - start
boost_ap = average_precision_score(yb_valid, bank_boost.predict_proba(Xb_valid)[:, 1])
print("XGBoost:", round(boost_seconds, 1), "seconds,  validation average precision", round(boost_ap, 3),
      " stopped at tree", bank_boost.best_iteration)

XGBoost lands at about **0.46** and the forest at about **0.457**. Tied again, on 33,908 customers this time, and the two took a similar number of seconds. What XGBoost did give us for free is the number of trees: it stopped itself at 651 instead of us guessing. On this data the models are not what is holding the score down. The features simply do not say much about who will say yes.

---

# Stacking, in One Paragraph

The last section of Geron chapter 6 is **stacking**. Instead of averaging or adding models, you train a final model whose input is the other models' predictions. It learns how much to trust each one.

In [ ]:
# RUN THIS. Stacking: a model that learns how to combine other models.
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

stack = StackingClassifier(
    estimators=[("forest", RandomForestClassifier(n_estimators=200, random_state=42)),
                ("boost", GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)),
                ("logistic", make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)))],
    final_estimator=LogisticRegression(max_iter=5000),   # the model that combines the three
    cv=5)                                                 # each member's predictions come from held out folds
stack.fit(X_train, y_train)

print("stacking, validation average precision:",
      round(average_precision_score(y_valid, stack.predict_proba(X_valid)[:, 1]), 3))

**0.668**, in the same tied group as everything else, for several times the training time and a lot more machinery. That is typical. Stacking wins competitions by third decimal places and is rarely worth it in a class project or a first real model. Know the word, know what it does, and reach for it last.

:::{.callout-note icon=false}
## Q6. Write this one out by hand

**a.** In your own words, what does the final estimator in a stack take as its input?

**b.** Stacking scored 0.668, the forest 0.667, boosting 0.673. Using what you learned in the repeated cross validation section, what should you say about these three numbers?
:::

---

# Putting It Together

| | bagging and forests (Day 6) | boosting (today) |
|---|---|---|
| how the trees are built | all at once, independently | one after another, each fixing the last |
| what each tree sees | its own bootstrap sample | what the model still gets wrong |
| more trees | never hurts, just slower | **can overfit**, so it is a real choice |
| main knobs | `n_estimators`, `max_features` | `learning_rate` with `n_estimators`, `max_depth` |
| how to set the number of trees | as many as you can afford | early stopping on a validation set |
| runs in parallel | yes | not really, each tree needs the one before |
| typical use | a strong, safe default | usually the best score on tables of numbers |

The recipe for today's family:

1. Start with `learning_rate=0.05` or `0.1` and a small `max_depth` (3 is a good first try).
2. Set `n_estimators` high and let **early stopping** pick the real number, watching a validation set.
3. Compare against a random forest with repeated cross validation, not one split.
4. Only claim a winner when the gap is bigger than the fold-to-fold wobble.
5. The test set stays closed until everything is decided.

# New Commands Today

| Command | What it does | The thing that trips people up |
|---|---|---|
| `GradientBoostingClassifier(n_estimators=, learning_rate=, max_depth=)` | trees added one at a time, each fitting the leftovers | more trees can make it worse. `learning_rate` and `n_estimators` trade off |
| `GradientBoostingRegressor(...)` | the same thing for a number instead of a class | `staged_predict` exists if you want the model after each tree |
| `AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=)` | the older method: re-weight the rows it gets wrong | the first argument is the model to boost. Stumps are the usual choice |
| `xgb.XGBClassifier(...)` | fast, regularized gradient boosting | a separate package (`import xgboost as xgb`), but it behaves like an sklearn model |
| `early_stopping_rounds=` with `eval_set=[(X_valid, y_valid)]` | stops adding trees when validation stops improving | `eval_set` is the **validation** set, never the test set. Read `best_iteration` afterwards |
| `RepeatedStratifiedKFold(n_splits=5, n_repeats=4)` | 20 fits over different shuffles | pair it with `cross_val_score` and look at the standard deviation, not just the mean |
| `StackingClassifier(estimators=[...], final_estimator=)` | a model that learns how to combine models | slow, and rarely worth it. `cv=5` keeps the members' predictions honest |

:::{.callout-tip icon=false}
## You Try: optional code

Nothing here is collected. Work through it if you want the idea to stick.

**1.** In the wine gradient boosting cell, try `max_depth=2` and `max_depth=8`. Which does better on validation? Boosting usually likes shallow trees, does the wine agree?

**2.** Give the XGBoost early stopping cell `learning_rate=0.01` instead of 0.05. Where does it stop now, and does the score change? This is the trade off from the learning rate section, on real data.

**3.** Run the repeated cross validation comparison again with `n_repeats=10`. Does the gap between the forest and the boosted model look any more real with 50 fits?

**4.** Add XGBoost to the stacking list. Does the stack improve? Was it worth the wait?
:::

# Before Next Class

1. In your lecture notes notebook, add your hand written notes and answers to the questions.
2. Do the **Day 7 practice problems** in `HW_day7.ipynb`.
3. **Weekly Homework 4** is due **Sunday 9/27 at 11:59pm**. It covers Day 7 and Day 8.
4. Read Geron chapter 2, all of it. It is the end to end project chapter, and Day 8 follows it.
5. Watch the Day 8 video on the class website.

Day 8 puts the whole thing together: one pipeline that takes raw data in one end and a tuned, evaluated model out the other. Everything since Day 2 has been a piece of that pipeline.

# Answers to the Q Boxes

Try every one of these by hand first. These are short summaries, not full answers, and the writing out is the part that does the work.

:::{.callout-note collapse="true"}
## Q1. What boosting does

**a.** Each new tree is trained on what the model so far still gets wrong, the leftovers or residuals. In a random forest every tree is trained on its own bootstrap sample of the original data and never looks at the other trees.

**b.** `y_fake` is a smooth curve plus random noise. A line that passes through nearly every point has fitted the noise, which is different for every new sample, so it will do worse on new points. That is the 500 tree panel.

**c.** Bagging. Its trees do not depend on each other, so a computer with eight cores can build eight at a time (that is what `n_jobs` does). Boosting has to finish one tree before it knows what the next one should fix.
:::

:::{.callout-note collapse="true"}
## Q2. Too many trees

**a.** A forest averages independent trees, so extra trees only make the average steadier. A boosted model keeps adding corrections to itself, and once the real pattern is used up the only thing left to correct is noise, so the extra trees fit noise.

**b.** The data itself carries noise with a standard deviation of 0.35. Even a model that knew the true curve exactly would miss each point by about that much. No model can predict a coin flip.

**c.** The training error at 2000 trees is 0.000, so you would conclude it is a perfect model. On new data it is the worst model in the table, 0.476 against 0.362 for 50 trees.
:::

:::{.callout-note collapse="true"}
## Q3. Learning rate

**a.** Raise `n_estimators`, by roughly the same factor you lowered the rate. In the table, 0.1 needed 50 trees and 0.01 needed 500.

**b.** Each tree is added at full strength, so the model lurches past the answer instead of creeping up on it, and later trees spend their time undoing earlier overshoots. Its best score came from a single tree.

**c.** A tiny learning rate with a huge number of trees still overfits, the curve just takes longer to turn upward, and every one of those trees costs time. The fix is not a smaller rate, it is stopping at the bottom of the curve.
:::

:::{.callout-note collapse="true"}
## Q4. Early stopping

**a.** It keeps going until 50 trees in a row have failed to improve the validation score, then stops and keeps the best one. Stopping at the first failure would quit far too early, because the score wobbles up and down from tree to tree even while it is still improving overall.

**b.** Stopping early is a **choice**, the choice of how many trees. Choices are made on validation data. If the test set made that choice, its score would include that choice and would no longer be an honest estimate.

**c.** `n_estimators=300` would give the same model, since it stopped at 193 on its own. `n_estimators=100` would cut it off before it was done, and you would never know, because it would simply run out of trees rather than stop.
:::

:::{.callout-note collapse="true"}
## Q5. Tied models

**a.** One validation score is a single sample: these particular 300 wines, of which only 41 are good. The mean over 20 fits uses every training wine as held out data at some point, so it averages out which wines happened to land where.

**b.** Say the two models were measured with a ruler that wobbles by about 0.085 every time you use it. A difference of 0.006 is far smaller than the wobble, so you cannot tell the models apart with it. You would need either a much bigger gap or a much steadier ruler.

**c.** Reasons for the forest: it has fewer knobs to get wrong, it trains in parallel, and out-of-bag scoring is free. Reasons for boosting: it usually scales better to bigger data, early stopping picks the tree count for you, and XGBoost is what the industry expects to see. Either is defensible when the scores are tied.
:::

:::{.callout-note collapse="true"}
## Q6. Stacking

**a.** The predictions of the member models. Each member makes a prediction for a row, and those predictions become the features the final estimator learns from, which is how it learns whom to trust when.

**b.** They are tied. All three sit inside the fold-to-fold wobble we measured, so ranking them by the third decimal place is reading noise. Pick on training time, simplicity, or what your team can maintain.
:::